# ML Assignment 2 — Credit Card Fraud Detection
No pre-trained model files. All training happens at runtime inside the Streamlit app.
This notebook is for local experimentation and verification only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../test_data.csv')
print('Shape:', df.shape)
print('Target distribution:')
print(df['is_fraud'].value_counts())
df.head()

In [ ]:
df = df.drop(columns=['transaction_id'])

bool_cols = ['is_foreign_transaction','is_new_merchant','used_vpn',
             'ip_country_mismatch','billing_shipping_mismatch','is_ai_generated_scam_attempt']
for c in bool_cols:
    df[c] = df[c].astype(int)

cat_cols     = ['merchant_category','card_type','auth_method','channel','device_type']
numeric_cols = [c for c in df.columns if c not in cat_cols + ['is_fraud']]

X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Fraud in test: {y_test.sum()} / {len(y_test)}')

In [ ]:
THRESH = 0.15

pre_scaled = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
])

pre_trees = ColumnTransformer([
    ('num', 'passthrough', numeric_cols),
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
])

pipelines = {
    'Logistic Regression': Pipeline([('pre', pre_scaled), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))]),
    'Decision Tree':       Pipeline([('pre', pre_trees),  ('clf', DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, class_weight='balanced', random_state=42))]),
    'kNN':                 Pipeline([('pre', pre_scaled), ('clf', KNeighborsClassifier(n_neighbors=5, weights='distance'))]),
    'Naive Bayes':         Pipeline([('pre', pre_scaled), ('clf', GaussianNB(var_smoothing=1e-8))]),
    'Random Forest':       Pipeline([('pre', pre_trees),  ('clf', RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', max_depth=12, random_state=42, n_jobs=-1))]),
}

results = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= THRESH).astype(int)
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'AUC':       round(roc_auc_score(y_test, y_prob), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 4),
        'MCC':       round(matthews_corrcoef(y_test, y_pred), 4),
        'CM':        confusion_matrix(y_test, y_pred).tolist(),
    }
    print(f"{name}: {results[name]}")

In [ ]:
df_res = pd.DataFrame(results).T.drop(columns=['CM'])
print('\nModel Comparison Table:')
display(df_res)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, _) in zip(axes, results.items()):
    cm = np.array(results[name]['CM'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No','Yes'], yticklabels=['No','Yes'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()